# Исследование данных о продажах компьютерных игр


**Автор:** Виктория Лентини

**Дата:** 22 октября 2026

## Описание проекта

Коллеги из команды игры «Секреты Темнолесья» изучают историю продаж игр в начале XXI века. Нам предоставили данные о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки.

**Цель проекта:** провести предобработку данных и подготовить срез за 2000–2013 годы для дальнейшего анализа.

**Задачи:**
1. Загрузить данные и познакомиться с ними.
2. Проверить данные на ошибки и провести предобработку: названия столбцов, типы данных, пропуски, дубликаты.
3. Отфильтровать игры по году выпуска (2000–2013 гг. включительно).
4. Категоризировать игры по пользовательским и экспертным оценкам.
5. Выделить топ-7 платформ по количеству выпущенных игр.

## Описание данных

Файл `/datasets/new_games.csv` содержит следующие поля:

- `Name` — название игры;
- `Platform` — название платформы;
- `Year of Release` — год выпуска игры;
- `Genre` — жанр игры;
- `NA sales` — продажи в Северной Америке (млн копий);
- `EU sales` — продажи в Европе (млн копий);
- `JP sales` — продажи в Японии (млн копий);
- `Other sales` — продажи в других странах (млн копий);
- `Critic Score` — оценка критиков (от 0 до 100);
- `User Score` — оценка пользователей (от 0 до 10);
- `Rating` — рейтинг организации ESRB (возрастная категория игры).

## Содержание

1. [Загрузка и знакомство с данными](#1-bullet)
2. [Предобработка данных](#2-bullet)
    * [Названия столбцов](#2.1-bullet)
    * [Типы данных](#2.2-bullet)
    * [Пропуски](#2.3-bullet)
    * [Дубликаты](#2.4-bullet)
    * [Промежуточный вывод по предобработке](#2.5-bullet)
3. [Фильтрация данных](#3-bullet)
4. [Категоризация данных](#4-bullet)
    * [Категории по оценкам пользователей и критиков](#4.1-bullet)
    * [Топ-7 платформ](#4.2-bullet)
    * [Промежуточный вывод по категоризации](#4.3-bullet)
5. [Общий вывод](#5-bullet)

<a id='1-bullet'></a>
## 1. Загрузка и знакомство с данными: импортируем необходимые библиотеки и загружаем данные 

In [1]:
# Импортируем библиотеку pandas
import pandas as pd

In [2]:
# Выгружаем данные из датасета new_games.csv в датафрейм df
df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

In [3]:
# Выводим информацию о датафрейме
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


In [4]:
# Выводим первые строки датафрейма на экран с помощью метода head(). По умолчанию метод выводит первые 5 строк
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


### Вывод по первичному знакомству с данными

- **Объём данных:** 16956 строк и 11 столбцов.
- **Соответствие описанию:** названия столбцов соответсвуют данному нам описанию. Все названия столбцов записаны с заглавных букв через пробел, в дальнейшем мы приведем их к snake_case.
- **Пропуски:** больше всего пропусков в оценках и рейтинге. Столбцы `Name`, `Genre` и `Year of Release` почти полностью заполнены.
- **Типы данных:** следующие столбцы требуют преобразования:
   - `Year of Release` (`float64`) — год должен быть целым числом (`int64`). Вещественный тип появился из-за пропусков, так как `int64` не может хранить `NaN`.
   - `EU sales` и `JP sales` (`str`) — продажи должны быть числовыми (`float64`). Предположительно, в столбцах есть нечисловые значения, из-за которых pandas не смог определить числовой тип.
   - `User Score` (`str`) — оценка должна быть числовой (`float64`). Скорее всего, причина в значении `tbd` (to be determined: оценка ещё не определена), но мы проверим эту гипотезу.

- **Другие особенности:** `info()` не считает значения вроде `tbd` пропусками, поэтому реальное число пропусков станет известно после преобразования типов.
- **План предобработки:** 
    1. привести названия столбцов к snake_case;
    2. выяснить, какие строки мешают числовому типу, и преобразовать `EU sales`, `JP sales`, `User Score`;
    3. обработать пропуски и привести `Year of Release` к целому типу;
    4. проверить и устранить дубликаты.

<a id='2-bullet'></a>
## 2. Предобработка данных

<a id='2.1-bullet'></a>
### 2.1 Еще раз просмотрим названия столбцов и приведем их к стилю snake_case

In [5]:
# Выводим названия столбцов
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')

In [6]:
# Меняем названия столбцов с помощью метода replace() 
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [7]:
# Посмотрим результат
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')

Все получилось. Теперь назвния столбцов приведены в правильную стилистику.

<a id='2.2-bullet'></a>
### 2.2 Типы данных

Теперь проверим типы данных и преобразуем их в корректные.

In [8]:
# Еще раз взглянем на типы данных
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  str    
 6   jp_sales         16956 non-null  str    
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  str    
 10  rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


Столбцы `eu_sales`, `jp_sales` и `user_score` имеют тип `str`, хотя должны быть числовыми. Найдём значения, которые этому мешают.

In [9]:
# Проверим все уникальные занчения в столбцe user_score и проверим нашу гипотезу о наличии значений `tbd`, которые могли повлиять на тип данных
df['user_score'].sort_values().unique()

<StringArray>
[  '0', '0.2', '0.3', '0.5', '0.6', '0.7', '0.9',   '1', '1.1', '1.2', '1.3',
 '1.4', '1.5', '1.6', '1.7', '1.8', '1.9',   '2', '2.1', '2.2', '2.3', '2.4',
 '2.5', '2.6', '2.7', '2.8', '2.9',   '3', '3.1', '3.2', '3.3', '3.4', '3.5',
 '3.6', '3.7', '3.8', '3.9',   '4', '4.1', '4.2', '4.3', '4.4', '4.5', '4.6',
 '4.7', '4.8', '4.9',   '5', '5.1', '5.2', '5.3', '5.4', '5.5', '5.6', '5.7',
 '5.8', '5.9',   '6', '6.1', '6.2', '6.3', '6.4', '6.5', '6.6', '6.7', '6.8',
 '6.9',   '7', '7.1', '7.2', '7.3', '7.4', '7.5', '7.6', '7.7', '7.8', '7.9',
   '8', '8.1', '8.2', '8.3', '8.4', '8.5', '8.6', '8.7', '8.8', '8.9',   '9',
 '9.1', '9.2', '9.3', '9.4', '9.5', '9.6', '9.7', 'tbd',   nan]
Length: 97, dtype: str

В результате нашлось значение `tbd`(предпоследнее с конца), а значит наша гипотеза из выводов по первичному знакомству с данными подтвердилась. Метод `sort_values()` cделал сортировку и поставил нечисловые значения в конец списка, так обнаружить нужный нам тип легче.

In [10]:
# Сколько значений 'tbd' (считаем до замены)
(df['user_score'] == 'tbd').sum()

np.int64(2464)

В `user_score` обнаружено 2 464 значения `tbd`, из-за них столбец имел тип `str`.

In [11]:
# Ищем нечисловые значения в eu_sales и jp_sales
for clmn in ['eu_sales', 'jp_sales']:
    bad = df.loc[pd.to_numeric(df[clmn], errors='coerce').isna() & df[clmn].notna(), clmn]
    print(clmn, '->', bad.unique(), '| штук:', len(bad))

eu_sales -> <StringArray>
['unknown']
Length: 1, dtype: str | штук: 6
jp_sales -> <StringArray>
['unknown']
Length: 1, dtype: str | штук: 4


Нашли 6 нечисловых значений в `eu_sales` и 4 в `jp_sales`.

In [12]:
# Теперь преобразуем столбцы в числовой тип, нечисловые значения становятся NaN благодря параметру 'coerce'
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.4 MB


Получилось. Теперь столбцы `eu_sales`, `jp_sales` и `user_score` имеют тип `float64`. 

**Промежуточный вывод по типам данных:**
- В `user_score` обнаружено 2 464 значения `tbd` (to be determined: оценка ещё не определена). Из-за них столбец имел тип `str`. Значения заменены на пропуски.
- В `eu_sales` и `jp_sales` были найдены нечисловые значения: 6 и 4 записи `'unknown'` соответственно. Заменены на пропуски.
- Столбцы `eu_sales`, `jp_sales` и `user_score` теперь приведены к типу `float64`.
- `year_of_release` останется вещественным, пока в нём есть пропуски. Приведём его к `int64` после их обработки, так как помним что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`.

<a id='2.3-bullet'></a>
### 2.3 Пропуски
Теперь посчитаем пропуски. Их стало больше, чем показывал `info()` изначально, так как нечисловые значения превратились в `NaN`.

In [13]:
# Считаем пропуски в каждом столбце: абсолютное количество и долю в процентах
missing = pd.DataFrame({
    'пропусков': df.isna().sum(),                       # количество пропусков
    'доля, %': (df.isna().mean() * 100).round(2)        # доля пропусков от всех строк
})
# Оставляем только столбцы с пропусками и сортируем по убыванию
missing[missing['пропусков'] > 0].sort_values('пропусков', ascending=False)     

,пропусков,"доля, %"
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01


**Промежуточный вывод по пропускам:**
Нужно решить что делать с пропусками в каждом столбце. Разберем возможные причины пропусков и подберем наиболее корректное решение:

| Столбец | Пропусков | Доля, % | Возможная причина | Решение |
|---|---|---|---|---|
| `name`, `genre` | 2 | 0,01 | ошибка при выгрузке | удалить |
| `year_of_release` | 275 | 1,62 | год не указан | удалить |
| `critic_score` | 8714 | 51,39 | игру не оценили | оставить |
| `user_score` | 9268 | 54,66 | игру не оценили, значение `tbd` | оставить |
| `rating` | 6871 | 40,52 | рейтинг не присвоен | заменить на `unknown` |
| `eu_sales`, `jp_sales` | 6 и 4 | 0,04 и 0,02 | нечисловые значения | удалить |

**Теперь объясним почему именно следующие решения наиболее подходящие в конкретной ситуации:**

- Cтолбцы `name`, `genre` - доля пропусков всего лишь 0,01% от общего количества данных, удаление при такой низкой доле допустимо;
- `year_of_release` - причина та же: доля пропусков достаточно мала (1,62%), а значит удаление допустимо;
- `critic_score` - оставить `NaN`: любая замена исказит распределение оценок, такая высокая доля (51,39%) говоит о том, что критики возможно просто не оценивали игру, а значит данных вовсе нет;
- `user_score` - выберем то же решение что и для `critic_score` - оставляем `NaN`, ничем не заменяем, так как доля слишком высокая (54,66%)
- `rating` - высокая доля пропусков (40,52%), можно заменить пропуски на значение `unknown`. В `rating` лежат текстовые категории (E, T, M и т. д.), считать по ним среднее не нужно, а следовательно значение-индикатор здесь ничего не исказит, а просто будет обозначать что рейтинг неизвестен;
- Столбцы `eu_sales`, `jp_sales` - доля пропусков всего 0,04% и 0,02%, их можно было бы заменить на среднее значение в зависимости от названия платформы и года выхода игры, как нам было предложено в задании, но удаление также не сильно исказит результаты и упростит работу.

**Приступим к непосредственной работе с пропусками**

In [14]:
# Удаляем строки с пропусками в названии, жанре, годе выпуска и продажах
df = df.dropna(subset=['name', 'genre', 'year_of_release', 'eu_sales', 'jp_sales']).copy()

In [15]:
# Rating: проверим все уникальные значения столбца и заменим NaN на'unknown'
print(df['rating'].unique())
df['rating'] = df['rating'].fillna('unknown')

<StringArray>
['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str


In [16]:
# Теперь год можно привести к целому типу
df['year_of_release'] = df['year_of_release'].astype('int64')

In [17]:
# Проверим результаты 
df.info()

<class 'pandas.DataFrame'>
Index: 16669 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16669 non-null  str    
 1   platform         16669 non-null  str    
 2   year_of_release  16669 non-null  int64  
 3   genre            16669 non-null  str    
 4   na_sales         16669 non-null  float64
 5   eu_sales         16669 non-null  float64
 6   jp_sales         16669 non-null  float64
 7   other_sales      16669 non-null  float64
 8   critic_score     8077 non-null   float64
 9   user_score       7550 non-null   float64
 10  rating           16669 non-null  str    
dtypes: float64(6), int64(1), str(4)
memory usage: 1.5 MB


**Итоги по пропускам и типам:** строки без названия, жанра и года удалены, пропуски в продажах также удалены; пропуски в `rating` заменены на `unknown`; пропуски в оценках оставлены, так как заполнение исказило бы данные. `year_of_release` приведён к `int64`.

<a id='2.4-bullet'></a>
### 2.4 Дубликаты
Сначала нормализуем текстовые данные и изучим уникальные значения, а затем найдем дубликаты.

In [18]:
# Нормализация текстовых столбцов
df['name'] = df['name'].str.strip().str.lower()
df['genre'] = df['genre'].str.strip().str.lower()
df['platform'] = df['platform'].str.strip().str.upper()
df['rating'] = df['rating'].str.strip().str.upper()

Теперь поищем неявные дубликаты среди столбцов с категориальными данными, например с названиями жанра игры, платформы, рейтинга и года выпуска. Уникальных значений в них не так много, поэтому результат можно оценить визуально:

In [19]:
# Ищем неявные дубликаты: опечатки, разное написание
print(df['platform'].unique())
print(df['genre'].unique())
print(df['rating'].unique())
print(sorted(df['year_of_release'].unique()))

<StringArray>
[ 'WII',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XONE',
 'WIIU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str
<StringArray>
[      'sports',     'platform',       'racing', 'role-playing',
       'puzzle',         'misc',      'shooter',   'simulation',
       'action',     'fighting',    'adventure',     'strategy']
Length: 12, dtype: str
<StringArray>
['E', 'UNKNOWN', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str
[np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003)

Опечаток и разного регистра нет, но есть один неявный дубликат: рейтинги K-A и E обозначают одно и то же. В описании данных в задании сказано: «До 1998 года рейтинг E был известен как Kids to Adults (K-A)». Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно, поэтому рейтинг с именем К-А не войдет в наш итоговый срез данных, но для чистоты все равно можем объединить их с Е:

In [20]:
# Смотрим сколько игр с K-A
print((df['rating'] == 'K-A').sum())

# Теперь объединим их с E
df['rating'] = df['rating'].replace('K-A', 'E')
print(df['rating'].unique())

3
<StringArray>
['E', 'UNKNOWN', 'M', 'T', 'E10+', 'AO', 'EC', 'RP']
Length: 8, dtype: str


Теперь не будет путаницы, все значения совпадают с актуальными категориями рейтинга ESRB.

Мы разобрались с неявными дубликатами, теперь посчитаем явные:

In [21]:
# Явные дубликаты: полные и по ключевым полям
print('Полных дубликатов:', df.duplicated().sum())
print('По названию, платформе и году:', df.duplicated(subset=['name', 'platform', 'year_of_release']).sum())

Полных дубликатов: 235
По названию, платформе и году: 236


**Что мы видим из результатов:**
- 235 полных дубликатов - это строки, совпадающие по всем столбцам. Их следует удалить.
- 236 дубликатов по названию, платформе и году. Это на одну больше, значит, есть строка, у которой название, платформа и год совпадают с другой. Проверим что это за строка:

In [22]:
# Сделаем ключ по которому будем искать строку, включаем три интересующих столбца
key = ['name', 'platform', 'year_of_release']

# Найдем строки, дублирующиеся по ключу, но не являющиеся полными дубликатами
df[df.duplicated(subset=key, keep=False) & ~df.duplicated(keep=False)] 

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
606,madden nfl 13,PS3,2012,sports,2.11,0.22,0.0,0.23,83.0,5.5,E
16465,madden nfl 13,PS3,2012,sports,0.00,0.01,0.0,0.00,83.0,5.5,E


Вторая строка очевидно кажется лишней, а потому удалим и ее тоже.

In [23]:
# Удаляем все дубликаты, включая найденную лишнюю строку
df = df.drop_duplicates(subset=key)
print('Осталось строк:', len(df))

Осталось строк: 16433


Проверим что все дубликаты были удалены: до удаления дубликатов в данных было 16669 строк, а 16669 − 236 = 16433. Расчет совпадает с результатом, значит удалены все 236 найденных дубликатов.

**Промежуточный вывод по дубликатам:**

- **Нормализация:** названия игр и жанры приведены к нижнему регистру, названия платформ и рейтинг — к верхнему, лишние пробелы по краям удалены.
- **Неявные дубликаты:** в столбце `rating` значения `K-A` и `E` обозначают один и тот же рейтинг (K-A — прежнее название E до 1998 года), поэтому `K-A` заменено на `E`. В столбцах `platform`, `genre` и `year_of_release` неявных дубликатов не обнаружено: написание значений единообразно, аномальных годов нет.
- **Явные дубликаты:**
    - найдено 235 полных дубликатов (строки совпадают по всем столбцам);
    - ещё одна строка совпала с другой по названию, платформе и году выпуска (Madden NFL 13, PS3, 2012), но отличалась продажами: у неё они близки к нулю. Её тоже считаем дубликатом.
- **Обработка:** удалены все 235 строк полных дубликатов и 1 с повторами по названию, платформе и году выпуска.
- **Результат:** удалено 236 строк (около 1,4% данных), после удаления осталось 16 433 строки.

<a id='2.5-bullet'></a>
### 2.5 Промежуточный вывод по предобработке

В ходе предобработки мы подготовили данные к дальнейшему анализу:
- **Названия столбцов:** приведены к стилю snake_case. 
-  **Типы данных:**
    - `eu_sales`, `jp_sales` и `user_score` приведены к типу `float64`. Нечисловые значения заменены на пропуски, включая 2 464 значения `tbd` в `user_score`;
    - `year_of_release` приведён к `int64` после удаления строк с пропусками.
- **Пропуски:**
   - удалены строки с пропусками в `name`, `genre`, `year_of_release`, `eu_sales` и `jp_sales`: без названия, жанра и года игру нельзя идентифицировать и отнести к периоду, а пропусков в продажах было всего в сумме 10 (менее 0,1%), поэтому логичнее было их удалить;
   - пропуски в `rating` заменены на значение `unknown`: это категориальный столбец, а потому значение-индикатор позволит не потерять эти игры при группировке;
   - пропуски в `critic_score` и `user_score` оставлены без изменений: их доля превышает 50%, а любая замена исказила бы распределение оценок;
- **Дубликаты:**
   - текстовые данные нормализованы (приведены в предложенный в задании регистр, лишние пробелы убраны);
   - неявные дубликаты: рейтинги `K-A` и `E` объединены в `E`;
   - удалено 236 явных дубликатов (235 полных совпадений и одна запись, совпадающая с другой по названию, платформе и году выпуска).
- **Всего удалено строк:** изначальное количество строк - *16956*. Количество строк после обработки - *16433*. Удалено 523 строки (3,08%): 287 из-за пропусков и 236 дубликатов. Основные пропуски сосредоточены в оценках и рейтинге; это важно учитывать при анализе, так как практически у половины игр оценок нет.

<a id='3-bullet'></a>
## 3. Фильтрация данных
Коллег интересует период с 2000 по 2013 год включительно.  Отберем данные по этому показателю и сохраним новый срез данных в отдельном датафрейме.

In [24]:
# Сделаем булеву маску для фильтрации 
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [25]:
# Проверим результат 
print(df_actual['year_of_release'].sort_values().unique())

[2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013]


Все нужные нам годы вошли в итоговый срез данных.

In [26]:
# Выведем информацию о новом срезе данных
df_actual.info()

<class 'pandas.DataFrame'>
Index: 12771 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12771 non-null  str    
 1   platform         12771 non-null  str    
 2   year_of_release  12771 non-null  int64  
 3   genre            12771 non-null  str    
 4   na_sales         12771 non-null  float64
 5   eu_sales         12771 non-null  float64
 6   jp_sales         12771 non-null  float64
 7   other_sales      12771 non-null  float64
 8   critic_score     7161 non-null   float64
 9   user_score       6475 non-null   float64
 10  rating           12771 non-null  str    
dtypes: float64(6), int64(1), str(4)
memory usage: 1.2 MB


Теперь наш срез данных выглядит так, значений в каждом столбце стало меньше, а значит фильтрация сработала и  мы можем приступать к категоризации. 

<a id='4-bullet'></a>
## 4. Категоризация данных

Нам нужно провести категоризацию данных и создать новые столбцы:

- `user_score_category`: **высокая** — от 8 до 10 включительно, **средняя** — от 3 до 8 (8 не входит), **низкая** — от 0 до 3 (3 не входит);
- `critic_score_category`: **высокая** — от 80 до 100 включительно, **средняя** — от 30 до 80 (80 не входит), **низкая** — от 0 до 30 (30 не входит).
  
(*Важное уточнение: игры без оценок не войдут в выборку, так как ранее мы оставили им значение `NaN`, а значит они не попадут ни в одну категорию*)

Затем мы выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [27]:
# Распредлим на категории оценки пользователей
def user_category(score):
    if score >= 8:
        return 'высокая'
    elif score >= 3:
        return 'средняя'
    elif score >= 0:
        return 'низкая'

df_actual['user_score_category'] = df_actual['user_score'].apply(user_category)

In [28]:
# Распредлим на категории оценки критиков
def critic_category(score):
    if score >= 80:
        return 'высокая'
    elif score >= 30:
        return 'средняя'
    elif score >= 0:
        return 'низкая'

df_actual['critic_score_category'] = df_actual['critic_score'].apply(critic_category)        

<a id='4.1-bullet'></a>
**Проверим как распределились оценки по категориям:** 

In [29]:
# Посмотрим результаты и дадим им табличный вид для наглядности
categories_table = pd.DataFrame({
    'оценки пользователей': df_actual['user_score_category'].value_counts(dropna=False),
    'оценки критиков': df_actual['critic_score_category'].value_counts(dropna=False)
})
categories_table

,оценки пользователей,оценки критиков
NaN,6296,5610
средняя,4077,5421
высокая,2282,1685
низкая,116,55


Теперь видим распределение всех игр в конкретных цифрах

(*NaN - игры без оценки*)

In [30]:
# Посмотрим доли только среди игр, у которых есть оценка пользователей
df_actual['user_score_category'].value_counts(normalize=True).round(3) * 100

user_score_category
средняя    63.0
высокая    35.2
низкая      1.8
Name: proportion, dtype: float64

Сразу видим что категория со средними оценками заметно преобладает

In [31]:
# Посмотрим доли только среди игр, у которых есть оценка критиков
df_actual['critic_score_category'].value_counts(normalize=True).round(3) * 100

critic_score_category
средняя    75.7
высокая    23.5
низкая      0.8
Name: proportion, dtype: float64

Здесь тоже преобладает категория со средними оценками, но вот количество высоких и низких оценок уже меньше

In [32]:
# Проверим итоговый срез данных, посмотрим добавились ли новые новые столбцы с категориями

df_actual.info()

<class 'pandas.DataFrame'>
Index: 12771 entries, 0 to 16954
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   name                   12771 non-null  str    
 1   platform               12771 non-null  str    
 2   year_of_release        12771 non-null  int64  
 3   genre                  12771 non-null  str    
 4   na_sales               12771 non-null  float64
 5   eu_sales               12771 non-null  float64
 6   jp_sales               12771 non-null  float64
 7   other_sales            12771 non-null  float64
 8   critic_score           7161 non-null   float64
 9   user_score             6475 non-null   float64
 10  rating                 12771 non-null  str    
 11  user_score_category    6475 non-null   str    
 12  critic_score_category  7161 non-null   str    
dtypes: float64(6), int64(1), str(6)
memory usage: 1.4 MB


Столбцы добавились, значит можем переходить к следующей части:

<a id='4.2-bullet'></a>
**Теперь создадим топ-7 платформ по количеству игр, выпущенных за весь актуальный период**

In [33]:
# Считаем количество значений по каждой платформе, выбираем только 7 первых с помощью метода head(7)
top_platforms = df_actual['platform'].value_counts().head(7)
top_platforms

platform
PS2     2126
DS      2117
WII     1275
PSP     1179
X360    1118
PS3     1086
GBA      810
Name: count, dtype: int64

In [34]:
# Посчитаем процент выпуска от всех игр для нашего топа
round(top_platforms.sum() / len(df_actual) * 100, 1)

np.float64(76.0)

<a id='4.3-bullet'></a>
**Промежуточный вывод по категоризации:**

В новый срез данных `df_actual` (игры 2000-2013 годов) добавлены два новых столбца:
- `user_score_category` — категория по оценке пользователей: высокая (от 8 до 10 включительно), средняя (от 3 до 8, не включая 8), низкая (от 0 до 3, не включая 3);
- `critic_score_category` — категория по оценке критиков: высокая (от 80 до 100 включительно), средняя (от 30 до 80, не включая 80), низкая (от 0 до 30, не включая 30).

| Категория | Оценки пользователей | Оценки критиков |
|---|---|---|
| Высокая | 2282 | 1685 |
| Средняя | 4077 | 5421 |
| Низкая | 116 | 55 |
| Без оценки | 6296 | 5610 |

**Выводы**:
- Игры без оценки в категории не вошли: у них нет значения для категоризации, поэтому в новых столбцах стоят пропуски.
- **Оценки пользователей и критиков** (доли посчитаны среди игр, у которых есть оценка):
    - преобладает **средняя** категория: 63% у пользователей и 75,7% игр у критиков;
    - высокую оценку получили 35,2% и 23,5% игр соответственно;
    - низкие оценки встречаются редко: всего 1,8% и 0,8% игр соответственно. 
- Отдельные наблюдения: Критики практически вдвое реже ставят низкие оценки, а также ставят меньше высоких оценок. Зато разница между игроками и критиками меняется в среднем сегменте оценок - там критики ставят больше средних оценок, чем пользователи. Получается, что оценки пользователей разбросаны шире, а критики чаще придерживаются выставления средних оценок.

Мы также сделали **Топ-7 платформ** по количеству выпущенных игр 

Больше всего игр за период 2000-2013 годов вышло на следующие платформы:

| Место | Платформа | Игр |
|---|---|---|
| 1 | PS2 | 2 126 |
| 2 | DS | 2 117 |
| 3 | WII | 1 275 |
| 4 | PSP | 1 179 |
| 5 | X360 | 1 118 |
| 6 | PS3 | 1 086 |
| 7 | GBA | 810 |

- Лидируют **PS2** и **DS**, разрыв между ними минимален - всего 9 игр. Они заметно опережают остальных: третья по счёту WII отстаёт от DS более чем на 800 игр.
- PS2 выпустила в 2,6 раза больше игр, чем замыкающая список GBA.
- В топ вошли как стационарные консоли (PS2, WII, X360, PS3), так и портативные (DS, PSP, GBA).
- Суммарно на эти семь платформ приходится 76% всех игр в срезе df_actual.

<a id='5-bullet'></a>
## 5. Общий вывод

В проекте были подготовлены данные о продажах компьютерных игр для дальнейшего изучения истории продаж в начале XXI века. Исходный набор содержал 16 956 строк и 11 столбцов.

### Что было сделано

1. **Знакомство с данными.** Набор столбцов соответствовал описанию. Выявлены неверные типы данных, пропуски в шести столбцах и названия столбцов не в стиле snake_case.

2. **Предобработка данных:**
    - **Названия столбцов** приведены к snake_case.
    - **Типы данных:** `eu_sales`, `jp_sales` и `user_score` приведены к `float64`, нечисловые значения заменены на пропуски (в `user_score` было 2 464 значения `tbd`). `year_of_release` приведён к `int64` после обработки пропусков.
    - **Пропуски:**
        - удалены строки без названия, жанра, года выпуска и с пропусками в продажах;
        - пропуски в `rating` заменены на `UNKNOWN`;
        - пропуски в `critic_score` и `user_score` оставлены, так как их доля превышает 50%, а любая замена исказила бы распределение оценок.
    - **Дубликаты:** рейтинги `K-A` и `E` объединены (K-A - прежнее название E), текстовые данные нормализованы по регистру. Удалено 236 явных дубликатов.
    - **Итого удалено** 523 строки (3,08% от исходных данных), осталось 16 433 строки.



3. **Фильтрация.** Отобраны игры, вышедшие с 2000 по 2013 год включительно. Срез сохранён в датафрейм `df_actual`.

4. **Категоризация.** В `df_actual` добавлены два новых поля:
    - `user_score_category` — категория по оценке пользователей;
    - `critic_score_category` — категория по оценке критиков.

Значения обоих полей: высокая, средняя, низкая (границы заданы условиями задания проекта). Игры без оценки в категории не вошли, у них в этих полях стоят пропуски.

5. **Топ-7 платформ** по количеству игр за 2000-2013 годы: PS2 (2 126), DS (2 117), WII (1 275), PSP (1 179), X360 (1 118), PS3 (1 086), GBA (810).

### Описание итогового среза и новых полей

`df_actual` содержит 12771 строк и 13 столбцов: исходные 11 и два новых поля с категориями оценок. Период охвата — с 2000 по 2013 год.

### Основные наблюдения

- Среди игр с оценкой пользователей преобладает средняя категория - 63%, высокие оценки получили 35,2% игр, низкие — лишь 1,8%.
- По оценкам критиков: также преобладает средняя категория - 75,7%, высокие оценки получили 23,5%, низкие - только 0,8%.
- Лидеры по количеству игр — PS2 и DS, разрыв между ними минимален (9 игр). Третья в списке WII отстаёт более чем на 800 игр.
- Пропуски в оценках и рейтинге - самая заметная особенность данных, с которыми пришлось работать в проекте: оценок нет у значительной части игр, а рейтинга чуть менее чем у половины.

### Рекомендации для дальнейшего анализа

- Поскольку коллеги изучают историю продаж игр, то стоит учитывать что у большого числа игр оценок нет, а значит такие игры в категории не входят, следовательно, стоит быть очень осторожными с выводами о взаимосвязи оценок и продаж.
- Игры с рейтингом `UNKNOWN` стоит рассматривать как отдельную группу и не терять при группировках.
- Низкие оценки пользователей встречаются редко, поэтому выводы по этой категории будут менее надёжными. Такая картина может быть связана как с тем, что действительно плохих игр выпускалось очень мало, либо многие из этих плохих игр даже не удостоились оценки. В любом случае, данных недостаточно для того чтобы сделать однозначные выводы.